In [2]:
!mkdir audio_samples

In [15]:
!pip install whisper-timestamped
!pip install pandas

In [16]:
import whisper_timestamped as whisper
import pandas as pd

model = whisper.load_model("medium")

In [28]:
def process_segments(result, file_path, threshold=0.80):
  rows = []
  for segment in result['segments']:
    words = segment.get('words', [])
    if not words:
      continue
    avg_conf = sum([w['confidence'] for w in words]) / len(words)
    text = segment['text'].strip()
    duration = segment['end'] - segment['start']
    if avg_conf < threshold:
      continue
    if duration > 20:
      continue
    if text.lower().count("no") > 5:
      continue
    if len(text.split()) < 3:
      continue

    rows.append({
        "file": file_path,
        "sentence_id": len(rows) + 1,
        "start": segment['start'],
        "end": segment['end'],
        "transcript": text
    })
    return pd.DataFrame(rows)

In [29]:
file_path = "audio_samples/038010_EIT-2A.mp3"
result = whisper.transcribe(
    model,
    file_path,
    language="es",
    temperature=0
)

df = process_segments(result, file_path)

 95%|█████████▍| 51963/54963 [01:03<00:03, 814.20frames/s] 


In [30]:
def merge_segments(df, time_gap=1.0):
  merged = []
  current = df.iloc[0].copy()
  for i in range(1, len(df)):
    row = df.iloc[i]
    if (row['transcript'] == current['transcript']) or (row['start'] - current['end'] < time_gap):
      current['end'] = row['end']
    else:
      merged.append(current)
      current = row.copy()

    merged.append(current)

    return pd.DataFrame(merged)

In [36]:
df_merged = merge_segments(df)
df_final = df_merged.reset_index(drop=True)
df_final['sentence_id'] = df_final.index + 1
df_final.to_csv("final_transcriptions.csv", index=False, encoding="utf-8-sig")
df_final

,file,sentence_id,start,end,transcript
0,audio_samples/038010_EIT-2A.mp3,1,81.58,82.44,Le llamaré mañana.
1,audio_samples/038010_EIT-2A.mp3,2,107.74,110.34,"A veces, toman su perro para un parque."
2,audio_samples/038010_EIT-2A.mp3,3,132.99,156.72,Vamos a jugar a la voleybal.
3,audio_samples/038010_EIT-2A.mp3,4,193.06,199.56,¿Qué dice usted que va a hacer hoy?
4,audio_samples/038010_EIT-2A.mp3,5,201.96,208.64,Dudo que sepa manejar muy bien
5,audio_samples/038010_EIT-2A.mp3,6,211.32,218.48,Las calles de esta ciudad son muy anchas
6,audio_samples/038010_EIT-2A.mp3,7,222.76,230.04,Puede que llueva mañana todo el día
7,audio_samples/038010_EIT-2A.mp3,8,233.74,243.08,Las casas son muy bonitas pero caras
8,audio_samples/038010_EIT-2A.mp3,9,245.74,247.92,Me gustan las películas que acaban bien
9,audio_samples/038010_EIT-2A.mp3,10,257.44,259.96,El chico con el que yo salgo es español
